In [21]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [22]:
from datasets import load_dataset
import pandas as pd

# Load the Bitext customer support dataset
intent_dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
)

print("Dataset loaded successfully!")
print(intent_dataset)


Dataset loaded successfully!
DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 26872
    })
})


In [23]:
# Inspect intent labels

train_df = pd.DataFrame(intent_dataset["train"])

print("Dataset shape:", train_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

print("\nNumber of unique intents:")
print(train_df["intent"].nunique())

print("\nIntent labels:")
print(sorted(train_df["intent"].unique()))


Dataset shape: (26872, 5)

Columns:
['flags', 'instruction', 'category', 'intent', 'response']

Number of unique intents:
27

Intent labels:
['cancel_order', 'change_order', 'change_shipping_address', 'check_cancellation_fee', 'check_invoice', 'check_payment_methods', 'check_refund_policy', 'complaint', 'contact_customer_service', 'contact_human_agent', 'create_account', 'delete_account', 'delivery_options', 'delivery_period', 'edit_account', 'get_invoice', 'get_refund', 'newsletter_subscription', 'payment_issue', 'place_order', 'recover_password', 'registration_problems', 'review', 'set_up_shipping_address', 'switch_account', 'track_order', 'track_refund']


In [24]:
print("\nIntent distribution:")
print(train_df["intent"].value_counts())



Intent distribution:
intent
contact_customer_service    1000
complaint                   1000
check_invoice               1000
switch_account              1000
edit_account                1000
contact_human_agent          999
check_payment_methods        999
delivery_period              999
newsletter_subscription      999
get_invoice                  999
payment_issue                999
registration_problems        999
cancel_order                 998
place_order                  998
track_refund                 998
change_order                 997
set_up_shipping_address      997
check_refund_policy          997
create_account               997
get_refund                   997
review                       997
delivery_options             995
delete_account               995
recover_password             995
track_order                  995
change_shipping_address      973
check_cancellation_fee       950
Name: count, dtype: int64


In [25]:
print([name for name in globals() if "dataset" in name.lower() or "df" in name.lower()])


['load_dataset', 'intent_dataset', 'train_df', 'TfidfVectorizer', 'intent_df']


In [27]:
# Basic data quality checks

print("Missing values:")
print(train_df.isnull().sum())

print("\nDuplicate rows:")
print(train_df.duplicated().sum())

print("\nDuplicate instructions:")
print(train_df["instruction"].duplicated().sum())

print("\nUnique intents:", train_df["intent"].nunique())

print("\nIntent distribution:")
print(train_df["intent"].value_counts())


Missing values:
flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64

Duplicate rows:
0

Duplicate instructions:
2237

Unique intents: 27

Intent distribution:
intent
contact_customer_service    1000
complaint                   1000
check_invoice               1000
switch_account              1000
edit_account                1000
contact_human_agent          999
check_payment_methods        999
delivery_period              999
newsletter_subscription      999
get_invoice                  999
payment_issue                999
registration_problems        999
cancel_order                 998
place_order                  998
track_refund                 998
change_order                 997
set_up_shipping_address      997
check_refund_policy          997
create_account               997
get_refund                   997
review                       997
delivery_options             995
delete_account               995
recover_password             

In [28]:
# Analyze duplicate instructions and check for conflicting intent labels

duplicate_mask = train_df["instruction"].duplicated(keep=False)

duplicate_df = train_df[duplicate_mask].copy()

print("Total rows:", len(train_df))
print("Rows involved in duplicate instructions:",
      len(duplicate_df))

print("Unique duplicated instructions:",
      duplicate_df["instruction"].nunique())

# Check whether the same instruction has different intent labels
conflicting_duplicates = (
    duplicate_df
    .groupby("instruction")["intent"]
    .nunique()
)

conflicting_duplicates = conflicting_duplicates[
    conflicting_duplicates > 1
]

print("Instructions with conflicting intents:",
      len(conflicting_duplicates))

# Show examples if conflicts exist
if len(conflicting_duplicates) > 0:
    print("\nExamples of conflicting duplicates:\n")

    example_instructions = conflicting_duplicates.index[:10]

    print(
        duplicate_df[
            duplicate_df["instruction"].isin(example_instructions)
        ][["instruction", "intent"]]
        .sort_values("instruction")
        .to_string(index=False)
    )


Total rows: 26872
Rows involved in duplicate instructions: 3226
Unique duplicated instructions: 989
Instructions with conflicting intents: 0


In [29]:
# Remove exact duplicate instructions safely
# Since no duplicated instruction has conflicting intent labels,
# we can keep one copy of each instruction.

intent_df = train_df[
    ["instruction", "intent", "response"]
].drop_duplicates(
    subset=["instruction"],
    keep="first"
).reset_index(drop=True)

print("Original samples:", len(train_df))
print("After deduplication:", len(intent_df))
print("Removed samples:", len(train_df) - len(intent_df))

print("\nNumber of unique intents:",
      intent_df["intent"].nunique())

print("\nIntent distribution after deduplication:")
print(intent_df["intent"].value_counts().sort_index())


Original samples: 26872
After deduplication: 24635
Removed samples: 2237

Number of unique intents: 27

Intent distribution after deduplication:
intent
cancel_order                 493
change_order                 870
change_shipping_address      973
check_cancellation_fee       950
check_invoice                921
check_payment_methods        999
check_refund_policy          997
complaint                   1000
contact_customer_service    1000
contact_human_agent          999
create_account               892
delete_account               918
delivery_options             660
delivery_period              999
edit_account                 777
get_invoice                  909
get_refund                   918
newsletter_subscription      999
payment_issue                999
place_order                  998
recover_password             995
registration_problems        999
review                       997
set_up_shipping_address      997
switch_account               862
track_order            

In [31]:
# Prepare clean Train / Validation / Test splits

from sklearn.model_selection import train_test_split

X = intent_df["instruction"].astype(str)
y = intent_df["intent"].astype(str)

# First split: 70% Train, 30% Temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: 15% Validation, 15% Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))

print("\nNumber of intents:", y.nunique())

print("\nTrain intent distribution:")
print(y_train.value_counts().sort_index())

print("\nValidation intent distribution:")
print(y_val.value_counts().sort_index())

print("\nTest intent distribution:")
print(y_test.value_counts().sort_index())


Train samples: 17244
Validation samples: 3695
Test samples: 3696

Number of intents: 27

Train intent distribution:
intent
cancel_order                345
change_order                609
change_shipping_address     681
check_cancellation_fee      665
check_invoice               645
check_payment_methods       699
check_refund_policy         698
complaint                   700
contact_customer_service    700
contact_human_agent         699
create_account              624
delete_account              643
delivery_options            462
delivery_period             699
edit_account                544
get_invoice                 636
get_refund                  643
newsletter_subscription     699
payment_issue               699
place_order                 699
recover_password            697
registration_problems       699
review                      698
set_up_shipping_address     698
switch_account              603
track_order                 565
track_refund                495
Name: count, 

In [32]:
# Character-level TF-IDF for Intent Classification

from sklearn.feature_extraction.text import TfidfVectorizer

char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=2,
    max_features=20000,
    sublinear_tf=True
)

X_train_char = char_vectorizer.fit_transform(X_train)
X_val_char = char_vectorizer.transform(X_val)
X_test_char = char_vectorizer.transform(X_test)

print("Character TF-IDF shapes:")
print("X_train_char:", X_train_char.shape)
print("X_val_char:", X_val_char.shape)
print("X_test_char:", X_test_char.shape)

print("\nVocabulary size:",
      len(char_vectorizer.vocabulary_))


Character TF-IDF shapes:
X_train_char: (17244, 17978)
X_val_char: (3695, 17978)
X_test_char: (3696, 17978)

Vocabulary size: 17978


In [33]:
# Train Character TF-IDF + LinearSVC Intent Classifier

from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

char_clf = LinearSVC(
    C=2.0,
    random_state=42
)

print("Starting Intent Classifier training...")

char_clf.fit(
    X_train_char,
    y_train
)

print("Intent Classifier training completed!")


Starting Intent Classifier training...
Intent Classifier training completed!


In [34]:
# Evaluate Intent Classifier on Validation Set

from sklearn.metrics import accuracy_score, f1_score

y_val_pred = char_clf.predict(X_val_char)

val_accuracy = accuracy_score(
    y_val,
    y_val_pred
)

val_macro_f1 = f1_score(
    y_val,
    y_val_pred,
    average="macro"
)

print("Intent Classifier - Validation Results")
print("=" * 60)
print("Validation Accuracy:", round(val_accuracy, 4))
print("Validation Macro F1:", round(val_macro_f1, 4))


Intent Classifier - Validation Results
Validation Accuracy: 0.9992
Validation Macro F1: 0.9991


In [35]:
# Detailed Validation Classification Report

from sklearn.metrics import classification_report

print("Intent Classifier - Validation Classification Report")
print("=" * 70)

print(
    classification_report(
        y_val,
        y_val_pred,
        digits=4
    )
)


Intent Classifier - Validation Classification Report
                          precision    recall  f1-score   support

            cancel_order     1.0000    1.0000    1.0000        74
            change_order     1.0000    0.9924    0.9962       131
 change_shipping_address     1.0000    1.0000    1.0000       146
  check_cancellation_fee     1.0000    1.0000    1.0000       142
           check_invoice     1.0000    1.0000    1.0000       138
   check_payment_methods     1.0000    1.0000    1.0000       150
     check_refund_policy     1.0000    1.0000    1.0000       149
               complaint     1.0000    1.0000    1.0000       150
contact_customer_service     1.0000    1.0000    1.0000       150
     contact_human_agent     1.0000    1.0000    1.0000       150
          create_account     1.0000    0.9925    0.9963       134
          delete_account     1.0000    1.0000    1.0000       137
        delivery_options     1.0000    1.0000    1.0000        99
         delivery_peri

In [36]:
# Final Test Evaluation

from sklearn.metrics import accuracy_score, f1_score, classification_report

y_test_pred = char_clf.predict(X_test_char)

test_accuracy = accuracy_score(
    y_test,
    y_test_pred
)

test_macro_f1 = f1_score(
    y_test,
    y_test_pred,
    average="macro"
)

print("Intent Classifier - Final Test Results")
print("=" * 70)
print("Test Accuracy:", round(test_accuracy, 4))
print("Test Macro F1:", round(test_macro_f1, 4))

print("\nFinal Test Classification Report:")
print(
    classification_report(
        y_test,
        y_test_pred,
        digits=4
    )
)


Intent Classifier - Final Test Results
Test Accuracy: 0.9984
Test Macro F1: 0.9984

Final Test Classification Report:
                          precision    recall  f1-score   support

            cancel_order     1.0000    1.0000    1.0000        74
            change_order     0.9923    0.9923    0.9923       130
 change_shipping_address     1.0000    1.0000    1.0000       146
  check_cancellation_fee     1.0000    1.0000    1.0000       143
           check_invoice     0.9787    1.0000    0.9892       138
   check_payment_methods     1.0000    1.0000    1.0000       150
     check_refund_policy     1.0000    1.0000    1.0000       150
               complaint     1.0000    1.0000    1.0000       150
contact_customer_service     1.0000    1.0000    1.0000       150
     contact_human_agent     1.0000    1.0000    1.0000       150
          create_account     1.0000    1.0000    1.0000       134
          delete_account     0.9928    1.0000    0.9964       138
        delivery_option